[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LSDtopotools/lsdtt3_notebooks/blob/main/chi_analysis/salmon_river_chi_concavity_knickpoints.ipynb)

# Chi analysis of the tributaries of the Salmon River at Mackay Bar, Idaho: concavity, k<sub>sn</sub> and knickpoints

The main Salmon River and its South Fork meet at Mackay Bar, deep in the canyons of central Idaho. The two rivers have cut down through the Idaho batholith, and the small tributaries draining into them have to keep up. Where they have not yet caught up, their channels steepen in **knickpoints** that move upstream over time.

In this notebook we use the two rivers as the **base level**: we take every complete basin that drains into them, and analyse all of them together.

The workflow is:
1. Install `lsdviztools3` (the plotting package, from PyPI) and the `lsdtt3` command-line tools (from the latest GitHub release).
2. Download a **10 m** USGS 3DEP DEM with `lsdtt-fetch-raster`, and make a hillshade.
3. Find every complete basin draining into the two rivers with `lsdtt-basin-extraction`.
4. Find the **concavity index** (m/n) with the **disorder method** in `lsdtt-chi-analysis`: test a range of m/n values in every basin and keep the one with the least disorder.
5. Rerun `lsdtt-chi-analysis` at that m/n to get the **channel steepness index** k<sub>sn</sub>: a basin map, stacked chi plots and a k<sub>sn</sub> map.
6. Find the **knickpoints**, the places where k<sub>sn</sub> changes, and plot the major ones.

Each `lsdtt3` program is controlled by a small **parameter file**: a text file of `key: value` lines. We write these files from Python and then run the programs with `!`.

## Setup

### First set up condacolab

**`condacolab.install()` restarts the Colab runtime.** You will see a message that the session crashed: this is expected. Wait for it to reconnect, then carry on with the next cell.

In [ ]:
!pip install -q condacolab

In [ ]:
import condacolab
condacolab.install()

### Install PyGMT

`lsdviztools3` draws its maps with PyGMT, which needs the GMT library. pip cannot install GMT, so we install PyGMT (and GMT with it) from conda-forge. This is the slowest step: repeated tests suggest it takes around 70 seconds.

In [ ]:
# mamba prints hundreds of lines, so we hide them and only show the end of
# the output if something goes wrong.
import subprocess

result = subprocess.run(["mamba", "install", "-y", "pygmt"], capture_output=True, text=True)
if result.returncode == 0:
    print("pygmt installed successfully.")
else:
    print("pygmt installation FAILED. The last lines of the output were:\n")
    print("\n".join((result.stdout + result.stderr).splitlines()[-30:]))

### Install lsdviztools3 from PyPI

`lsdviztools3` needs Python 3.12 or later; the next cell prints the version this runtime has.

In [ ]:
import sys
print(sys.version)
!pip install -q "lsdviztools3>=0.4.2"
!lsdviz3-chi --help | head -n 3

### Get the lsdtt3 command-line tools

The Linux package of the latest [lsdtt3 release](https://github.com/LSDtopotools/lsdtt3/releases/latest). `releases/latest/download/` always points at the newest version.

In [ ]:
!wget -q https://github.com/LSDtopotools/lsdtt3/releases/latest/download/lsdtt3-backend-linux-x86_64-core.tar.gz
!tar -xzf lsdtt3-backend-linux-x86_64-core.tar.gz

Now we tell the system where to find the `lsdtt3` programs and the libraries they need. If your runtime restarts later, run this cell again.

In [ ]:
import glob
import os

root = sorted(glob.glob("/content/lsdtt3-backend-linux-x86_64-core-v*"))[-1]
print("Using", root)
os.environ["PATH"] = f"{root}/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = f"{root}/lib:" + os.environ.get("LD_LIBRARY_PATH", "")

# Do NOT set GDAL_DATA / PROJ_DATA / PROJ_LIB here. The lsdtt3 programs find
# their own bundled PROJ and GDAL data, and pointing the Python session at the
# backend's proj.db breaks rasterio and pyproj, which lsdviztools3 uses.

Check that it works: this prints the version of the tools.

In [ ]:
!lsdtt-chi-analysis -v

## Step 1: Download a 10 m DEM and make a hillshade

`lsdtt-fetch-raster` downloads a DEM for a box given in longitude and latitude. Here we use

* `dem_source: us_3dep`: the USGS 3D Elevation Program seamless DEM. It is a **bare-earth terrain model**, covers the United States, needs no account or key, and its default product is **10 m** (1/3 arc-second). We do not set `grid_spacing`, so the output stays at the native 10 m.
* We do not set `target_epsg` either: the tool picks the UTM zone from the centre of the box (here UTM zone 11N), so distances are in metres.

The box is about 27 km east–west by 23 km north–south, so at 10 m it is roughly 2700 by 2300 pixels. The main Salmon River enters from the east, the South Fork from the south, and they leave together across the western edge.

In [ ]:
# Lower left (south-west) and upper right (north-east) corners, as copied from Google Maps
lower_left = (45.30727069796397, -115.73589184958999)
upper_right = (45.513595114652674, -115.39652451446072)

with open("salmon_fetch.param", "w") as f:
    f.write("write_fname: salmon\n")
    f.write(f"west: {lower_left[1]}\n")
    f.write(f"east: {upper_right[1]}\n")
    f.write(f"south: {lower_left[0]}\n")
    f.write(f"north: {upper_right[0]}\n")
    f.write("dem_source: us_3dep\n")   # USGS 3DEP, 10 m bare earth

!lsdtt-fetch-raster ./ salmon_fetch.param
!ls -lh salmon*

In [ ]:
with open("salmon_hillshade.param", "w") as f:
    f.write("read_fname: salmon_DEM.tif\n")
    f.write("write_fname: salmon_hs\n")
    f.write("write_output_format: gtiff\n")
    f.write("write_hillshade_raster: true\n")

!lsdtt-raster-preprocessing ./ salmon_hillshade.param

In [ ]:
from IPython.display import Image, display

from lsdviztools3.render.hillshade import render_hillshade
from lsdviztools3.render.style import MapStyle
from lsdviztools3.render.base import save_figure

hs_style = MapStyle(cmap="gray", transparent=False)
fig = render_hillshade("salmon_hs_hillshade.tif", coordinates="local", shade=False, style=hs_style)
save_figure(fig, "salmon_hillshade.png", style=hs_style)
display(Image("salmon_hillshade.png", width=700))

### Where is this?

`lsdviz3-basemap` draws a **locator map**: the region around the DEM, with the state and country borders, rivers and lakes, shaded relief (`--relief`, downloaded from the GMT server), and the DEM's footprint in red. `--mode global` draws a globe instead.

In [ ]:
!lsdviz3-basemap --in salmon_DEM.tif --out salmon_locator.png --mode regional --relief --figure-size 10c

display(Image("salmon_locator.png", width=500))

## Step 2: The basins draining into the two rivers

`lsdtt-basin-extraction` with `extract_side_tributary_basins_from_points: true` starts from **seed points**, one on each river where it enters the box, and follows each river downstream (D8 flow routing) to the edge of the DEM. Every tributary that joins those two traced rivers is a candidate basin. It keeps a basin only if

* it is **complete** (`reject_incomplete_side_tributary_basins: true`): no part of it touches the edge of the DEM or nodata. A basin cut by the edge is missing some of its drainage area, so its chi values would be wrong;
* it is at least `side_tributary_minimum_contributing_pixels` in size: **50000 pixels = 5 km²** on a 10 m DEM.

The seed points do not have to be exactly on the river: the trace from a point on the valley side runs straight down into it.

The tool writes the basin **outlets** (`salmon_basins_side_tributary_outlets.fgb`), one pixel above where each tributary joins its river, and their outlines. The outlets file goes straight into `lsdtt-chi-analysis` in the next steps.

In [ ]:
import geopandas as gpd
import pandas as pd

# One seed on each river, where it enters the box
seeds = pd.DataFrame({"river": ["Salmon River", "South Fork Salmon River"],
                      "latitude": [45.4580, 45.3264],
                      "longitude": [-115.4269, -115.5010]})
gpd.GeoDataFrame(seeds, geometry=gpd.points_from_xy(seeds["longitude"], seeds["latitude"]),
                 crs="EPSG:4326").to_file("salmon_seeds.fgb")

with open("salmon_basins.param", "w") as f:
    f.write("read_fname: salmon_DEM.tif\n")
    f.write("write_fname: salmon_basins\n")
    f.write("threshold_contributing_pixels: 2000\n")
    f.write("extract_side_tributary_basins_from_points: true\n")
    f.write("side_tributary_points_fname: salmon_seeds.fgb\n")
    f.write("side_tributary_minimum_contributing_pixels: 50000\n")   # 5 km2 on a 10 m DEM
    f.write("reject_incomplete_side_tributary_basins: true\n")
    f.write("side_tributary_write_vector_format: flatgeobuf\n")

!lsdtt-basin-extraction ./ salmon_basins.param

In [ ]:
outlines = gpd.read_file("salmon_basins_side_tributary_basin_outlines.fgb")
print(f"{len(outlines)} complete basins of 5 km2 or more drain into the two rivers")
(outlines.area / 1e6).describe().round(1).rename("area (km2)")

## Step 3: The best concavity from a disorder scan

### How it works (Mudd et al., 2018)

Chi (χ) is a coordinate along each channel that integrates drainage area upstream from the outlet:

```
chi = integral from the outlet of (A0 / A)^(m/n) dx
```

where A is drainage area, A0 a reference area (1 m²) and **m/n the concavity index**. If a basin is in steady state, with uniform uplift and erodibility, and m/n is right, **every channel in the basin collapses onto one line** in a plot of elevation against chi. With the wrong m/n the tributaries sit above or below the main stem.

The **disorder** measures how far the channels are from collapsing onto one curve. Sort all the channel pixels of a basin by elevation; if chi only ever increases with elevation, the channels are perfectly collapsed. Every reversal adds to the disorder. lsdtt3 computes it for each m/n value in a range (`mn_min` to `mn_max` in steps of `mn_step`); in each basin, the m/n with the **lowest disorder** is the best estimate.

### How certain is it?

A basin's disorder depends on which tributaries it happens to have. With `run_bootstrap: true` the tool resamples each basin's side branches (always keeping the main stem) `bootstrap_replicates` times and recomputes the disorder at every m/n. That gives a 95 % interval for each m/n. An m/n value whose interval overlaps that of the best one cannot be told apart from it.

### The settings

* `basin_outlet_fname`: the outlets from Step 2. `basin_outlet_snapping_method: none` uses them as they are: they are already on the tributaries, and snapping downstream would move them onto the Salmon River.
* `n_largest_basins: 0` keeps every basin.
* `threshold_contributing_pixels: 2000`: chi is computed on channels with at least 0.2 km² of drainage area.

This run writes `salmon_disorder_chi_disorder_sweep.csv`: one row per basin and m/n value. It takes a few minutes.

In [ ]:
# The settings shared by both chi runs
common = {
    "read_fname": "salmon_DEM.tif",
    "basin_outlet_fname": "salmon_basins_side_tributary_outlets.fgb",
    "basin_outlet_snapping_method": "none",
    "threshold_contributing_pixels": 2000,     # 0.2 km2 on a 10 m DEM
    "n_largest_basins": 0,                     # keep every basin
    "fit_chi_networks": "true",
    "fit_critical_divergence_metres": 5,       # see Step 4
}


def write_param(fname, settings):
    """Write a dictionary of settings as an lsdtt3 parameter file."""
    with open(fname, "w") as f:
        for key, value in settings.items():
            f.write(f"{key}: {value}\n")


sweep_settings = dict(common)
sweep_settings.update({
    "write_fname": "salmon_disorder",
    "mn_min": 0.15,
    "mn_max": 0.85,
    "mn_step": 0.05,
    "run_bootstrap": "true",
    "bootstrap_replicates": 100,
    # Only the disorder is needed from this run, not the per-m/n fit tables
    "write_fit_tables": "false",
    "write_chi_tributary_table": "false",
    "write_chi_knickpoint_table": "false",
    "write_ksn_basin_summary": "false",
})
write_param("salmon_disorder.param", sweep_settings)

!lsdtt-chi-analysis ./ salmon_disorder.param

For each basin: the m/n with the lowest disorder, and the range of m/n values whose bootstrap interval overlaps it.

In [ ]:
sweep = pd.read_csv("salmon_disorder_chi_disorder_sweep.csv")
sweep["m_over_n"] = sweep["m_over_n"].round(3)

rows = []
for basin, b in sweep.groupby("basin_index"):
    best = b.loc[b["full_network_disorder"].idxmin()]
    same = b[(b["bootstrap_percentile_2p5"] <= best["bootstrap_percentile_97p5"]) &
             (b["bootstrap_percentile_97p5"] >= best["bootstrap_percentile_2p5"])]
    rows.append({"basin_index": basin, "best_m_over_n": best["m_over_n"],
                 "indistinguishable_from": same["m_over_n"].min(),
                 "indistinguishable_to": same["m_over_n"].max(),
                 "n_side_branches": int(best["n_side_branches"])})
best_by_basin = pd.DataFrame(rows)

# One concavity for the whole area: the median of the basins' best values,
# rounded to the nearest value that was tested
tested = sweep["m_over_n"].unique()
median_best = best_by_basin["best_m_over_n"].median()
best_mn = float(tested[abs(tested - median_best).argmin()])
print(f"Median of the basins' best m/n: {median_best:.3f}; used from here on: {best_mn}")
best_by_basin

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

# Each basin's disorder, divided by its own minimum, so the curves can share an axis
for basin, b in sweep.groupby("basin_index"):
    ax1.plot(b["m_over_n"], b["full_network_disorder"] / b["full_network_disorder"].min(),
             "-", lw=1, alpha=0.7)
ax1.axvline(best_mn, color="k", lw=1.5, ls="--")
ax1.set_xlabel("m/n")
ax1.set_ylabel("Disorder / basin minimum")
ax1.set_title("Disorder in each basin")

# The best m/n of each basin, with the range it cannot be told apart from
order = best_by_basin.sort_values("best_m_over_n")
y = range(len(order))
ax2.hlines(y, order["indistinguishable_from"], order["indistinguishable_to"], color="tab:blue", lw=3, alpha=0.4)
ax2.plot(order["best_m_over_n"], y, "o", color="tab:blue")
ax2.axvline(best_mn, color="k", lw=1.5, ls="--", label=f"m/n used = {best_mn}")
ax2.set_yticks(list(y))
ax2.set_yticklabels([f"basin {i}" for i in order["basin_index"]])
ax2.set_xlabel("Best m/n (bar: bootstrap range)")
ax2.set_title("Best m/n in each basin")
ax2.legend()
plt.tight_layout()
plt.show()

**Questions:** do the basins agree on a concavity? Are the bootstrap ranges narrow or wide? The method assumes uniform uplift and erodibility and a network near steady state. A basin with a knickpoint migrating through it is **not** at steady state: its channels above and below the knickpoint do not collapse onto one line at any m/n, which can make its disorder curve flat or its minimum misleading.

## Step 4: k<sub>sn</sub> at the best concavity

Now we run `lsdtt-chi-analysis` again at the chosen m/n only (`m_over_n`). It fits each channel with straight segments in chi–elevation space; the gradient of each segment is its **normalised channel steepness index** k<sub>sn</sub> (with A0 = 1 m², the units are m<sup>2m/n</sup>).

The fit splits a channel wherever the data are more than `fit_critical_divergence_metres` from the line. 5 m suits a 10 m DEM: smaller values give more, shorter segments that follow the noise in the DEM; larger values keep only the big changes.

This run also writes

* the basin polygons and chi data maps (`write_chi_data_maps`), for the basin map;
* the fitted segments as lines (`write_chi_fit_segment_lines`), for the k<sub>sn</sub> and knickpoint maps;
* the knickpoint tables used in Step 5.

In [ ]:
mn_suffix = f"{best_mn:.3f}".replace(".", "p")   # how lsdtt3 writes m/n in file names, e.g. 0p450

ksn_settings = dict(common)
ksn_settings.update({
    "write_fname": "salmon_ksn",
    "m_over_n": best_mn,
    "write_chi_data_maps": "true",
    "chi_data_maps_write_vector_format": "flatgeobuf",
    "write_chi_fit_segment_lines": "true",
    "write_vector_format": "flatgeobuf",
    "write_chi_knickpoint_table": "true",
})
write_param("salmon_ksn.param", ksn_settings)

!lsdtt-chi-analysis ./ salmon_ksn.param

### Basin map

The basins, numbered by `basin_key`. These are the same numbers as `basin_index` in the tables and the labels in the chi plots.

In [ ]:
!lsdviz3-basins --base salmon_hs_hillshade.tif --basins salmon_ksn_chi_basin_polygons.fgb \
    --label-col basin_key --out salmon_basins_map.png

display(Image("salmon_basins_map.png", width=800))

### Stacked chi plots and the k<sub>sn</sub> map

`lsdviz3-chi` draws the fitted network of every basin coloured by k<sub>sn</sub> over a hillshade, and a **stacked chi plot**: the chi–elevation profiles of all the basins, offset so they do not overlap, with their fitted segments. Add `--per-basin` for one plot per basin as well.

In [ ]:
!lsdviz3-chi --data-dir . --prefix salmon_ksn --out-prefix salmon --m-over-n {best_mn} \
    --dem salmon_DEM.tif --out-dir .

for png in ["salmon_ksn_map.png"] + sorted(glob.glob("salmon_chi*.png")):
    display(Image(png, width=800))

### k<sub>sn</sub> in classes

The same map with k<sub>sn</sub> in **classes** instead of a continuous colour scale, and a legend of the classes:

* `--ksn-classes natural_breaks` finds the breaks in the data (Jenks natural breaks, worked out in log10 k<sub>sn</sub>); `--n-classes` sets how many. Other choices are `equal_interval`, `quantile`, or the breaks themselves, e.g. `--ksn-classes 20,50,100,200`.
* `--locator regional` adds a small locator map in a corner (`--locator-position`).

In [ ]:
!lsdviz3-chi --data-dir . --prefix salmon_ksn --out-prefix salmon_classes --m-over-n {best_mn} \
    --dem salmon_DEM.tif --out-dir . --no-stacked \
    --ksn-classes natural_breaks --n-classes 5 --locator regional --locator-position BR

display(Image("salmon_classes_ksn_map.png", width=800))

In [ ]:
summary = pd.read_csv(f"salmon_ksn_chi_ksn_basin_summary_mn_{mn_suffix}.csv")
summary.round(1)

## Step 5: Knickpoints

A **knickpoint candidate** is any place where two fitted segments meet. `salmon_ksn_chi_knickpoints_mn_*.csv` lists them, with

| Column | Meaning |
|---|---|
| `basin_index` | Which basin |
| `ksn_upstream`, `ksn_downstream` | k<sub>sn</sub> of the segments above and below |
| `delta_ksn` | upstream minus downstream: **positive where the channel steepens upstream** |
| `log_ratio_ksn` | log10 of their ratio |
| `at_junction` | 1 at a tributary mouth, where a tributary meets its parent channel |
| `chi`, `elevation`, `flow_distance_m`, `drainage_area_m2`, `x`, `y` | where it is |

Every segment boundary is a candidate, so most are small. The tool does not pick the major ones for you: `salmon_ksn_chi_knickpoint_thresholds_mn_*.csv` **suggests thresholds** from the distribution of |`delta_ksn`| and |`log_ratio_ksn`| over the candidates that are not at tributary mouths: the 90th and 95th percentiles (`p90`, `p95`), the median plus three median absolute deviations (`median_plus_3mad`), and Otsu's threshold (`otsu`), which best splits the values into two groups. `n_above` is how many candidates each one keeps.

In [ ]:
knickpoints = pd.read_csv(f"salmon_ksn_chi_knickpoints_mn_{mn_suffix}.csv")
thresholds = pd.read_csv(f"salmon_ksn_chi_knickpoint_thresholds_mn_{mn_suffix}.csv")
print(f"{len(knickpoints)} knickpoint candidates, {int(knickpoints['at_junction'].sum())} of them at tributary mouths")
thresholds

`lsdviz3-knickpoints` picks the major knickpoints with one of these thresholds (`--threshold`, here the 95th percentile of |`delta_ksn`|) and draws:

* the distribution of the k<sub>sn</sub> changes, with the suggested thresholds and the one chosen;
* the major knickpoints on the map, over the k<sub>sn</sub> network;
* the major knickpoints on the stacked chi profiles;
* where they sit along chi.

It also writes the selected knickpoints to `salmon_major_knickpoints.csv`. Changes at tributary mouths are left out (add `--include-junctions` to count them): a step in k<sub>sn</sub> where a tributary joins is usually just the tributary being steeper than its parent, not a knickpoint moving up the channel.

In [ ]:
!lsdviz3-knickpoints --data-dir . --prefix salmon_ksn --out-prefix salmon --m-over-n {best_mn} \
    --dem salmon_DEM.tif --out-dir . --threshold p95

for png in ["salmon_knickpoint_distribution.png", "salmon_knickpoint_map.png",
            "salmon_knickpoint_profile.png", "salmon_knickpoints_along_chi.png"]:
    if os.path.exists(png):
        display(Image(png, width=800))

### Where are the major knickpoints?

If the Salmon River and its South Fork cut down, the signal travels up every tributary. In tributaries with the same uplift and erodibility, knickpoints that started at the same time tend to be found at **similar elevations** (or similar chi), even though the tributaries differ in size. Here are the major knickpoints of each basin by elevation:

In [ ]:
major = pd.read_csv("salmon_major_knickpoints.csv")
print(f"{len(major)} major knickpoints in {major['basin_index'].nunique()} basins")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
steep = major["delta_ksn"] > 0
ax1.scatter(major.loc[steep, "basin_index"], major.loc[steep, "elevation"], c="tab:red",
            label="steepens upstream")
ax1.scatter(major.loc[~steep, "basin_index"], major.loc[~steep, "elevation"], c="tab:blue",
            label="shallows upstream")
ax1.set_xlabel("basin_index")
ax1.set_ylabel("Elevation (m)")
ax1.set_title("Major knickpoints in each basin")
ax1.legend()

ax2.hist(major["elevation"], bins=20, orientation="horizontal", color="grey")
ax2.set_xlabel("Number of major knickpoints")
ax2.set_ylabel("Elevation (m)")
ax2.set_title("All basins")
plt.tight_layout()
plt.show()

cols = [c for c in ["basin_index", "elevation", "chi", "flow_distance_m", "drainage_area_m2",
                    "ksn_downstream", "ksn_upstream", "delta_ksn"] if c in major.columns]
major.sort_values(["basin_index", "elevation"])[cols].round(1)

**Questions:**

* Do the knickpoints that steepen upstream cluster at one elevation, or at a few? Compare that elevation with the height of the main rivers at the outlets.
* Are some basins missing knickpoints? Small basins may have been passed through already, or their channels may be too short for the fit to split.
* Look at the knickpoint map and the hillshade together. Do the major knickpoints sit where the valleys change shape, for example where a narrow inner gorge begins?

## Things to try

* Change `--threshold` to `otsu` or `median_plus_3mad`, or give a number. How many knickpoints survive each?
* Use `--measure log_ratio` to select by the ratio of k<sub>sn</sub> rather than the difference: a change from 50 to 100 then counts as much as one from 200 to 400.
* Use `--direction steepening` to keep only knickpoints where the channel steepens upstream.
* Give the k<sub>sn</sub> classes yourself, e.g. `--ksn-classes 20,50,100,200`, or use `--cmap tab10` for distinct colours instead of a gradient. The same options work on `lsdviz3-knickpoints`.
* Change `fit_critical_divergence_metres` (for example to 3 or 10) in `common`, and rerun Steps 3–5. How do the segments and knickpoints change?
* Change `side_tributary_minimum_contributing_pixels` in Step 2 to 20000 (2 km²) to include smaller basins.
* Narrow the disorder scan around the best value with a smaller `mn_step`, for example 0.3 to 0.6 in steps of 0.02.

## Every run is recorded

Each program writes `<write_fname>_ingested_params.param`: every parameter it used. You can run it again as it is to repeat the run. Keep these files with your results.

## References

* Perron, J. T. and Royden, L. (2013). An integral approach to bedrock river profile analysis. *Earth Surface Processes and Landforms*, 38(6), 570–576. https://doi.org/10.1002/esp.3302
* Mudd, S. M., Clubb, F. J., Gailleton, B. and Hurst, M. D. (2018). How concave are river channels? *Earth Surface Dynamics*, 6, 505–523. https://doi.org/10.5194/esurf-6-505-2018
* Hergarten, S., Robl, J. and Stüwe, K. (2016). Tectonic geomorphology at small catchment sizes – extensions of the stream-power approach and the χ method. *Earth Surface Dynamics*, 4(1), 1–9. https://doi.org/10.5194/esurf-4-1-2016
* The lsdtt3 documentation of the method: https://lsdtopotools.github.io/lsdtt3/design_and_algorithms/chi_analysis.html

## Data

Always cite the data you use. This notebook downloads:

* **DEM:** U.S. Geological Survey 3D Elevation Program (3DEP), 1/3 arc-second (about 10 m) seamless DEM, downloaded by `lsdtt-fetch-raster` (`dem_source: us_3dep`). Public domain; credit the U.S. Geological Survey: https://www.usgs.gov/3d-elevation-program
* **Locator map:** drawn with GMT through PyGMT, from GMT's coastline, border and river data (GSHHG), with shaded relief from GMT's `earth_relief` dataset. Wessel, P., Luis, J. F., Uieda, L., Scharroo, R., Wobbe, F., Smith, W. H. F. and Tian, D. (2019). The Generic Mapping Tools version 6. *Geochemistry, Geophysics, Geosystems*, 20, 5556–5564. https://doi.org/10.1029/2019GC008515
